In [ ]:
from gpaw.new.ase_interface import GPAW
from ase import Atoms
import numpy as np
from gpaw import FermiDirac
from ase.visualize import view

## Primitive cell

In [2]:
#building promitive cell of MnI2

#Lattice parameters:
a1 = 4.36 #Å
a2 = a1


primitive = Atoms(
    symbols='MnI2',
    cell=[a1, a2, 1.0, 90, 90, 120], # c=1.0 temporarily
    pbc=[True, True, False],
    scaled_positions=[
        [0.0, 0.0, 0.0],       # Mn
        [1/3, 2/3, 0.38],      # I1 (relative z)
        [2/3, 1/3, -0.38]      # I2 (relative z)
    ]
)

primitive.center(axis=2, vacuum=15.0)

for i, atom in enumerate(primitive):
    print(f"Atom {i} ({atom.symbol}): {atom.scaled_position}")


Atom 0 (Mn): [0.  0.  0.5]
Atom 1 (I): [0.33333333 0.66666667 0.51235371]
Atom 2 (I): [0.66666667 0.33333333 0.48764629]


## First attempt at magnetic supercell

simple (3,1,1) repetition.

In [ ]:
# #Supercell
# supercell = primitive * (3, 1, 1)
# print(f"Supercell contains: {supercell.get_chemical_formula()}")
# print(f'Number of atoms: {len(supercell)}')

Supercell contains: I6Mn3
Number of atoms: 9


In [ ]:
# # --- 3. Define the Spiral Magnetic Structure ---
# # Mn atoms are at indices 0, 3, 6 in the supercell (every 3rd atom starting at 0)
# # The spiral rotates by 120 degrees (2*pi/3) for each step along the spiral direction.
# # The article states the order is coplanar. Let's assume the spiral plane is the xy-plane.

# magmoms = np.zeros((len(supercell), 3))
# m = 4.5 # Mn2+ is typically high spin S=5/2 -> ~5 muB, DFT often yields ~4-5

# # Identify Mn indices (Atomic number 25)
# mn_indices = [i for i, atom in enumerate(supercell) if atom.symbol == 'Mn']

# assert len(mn_indices)==3, "should have 3 MN atoms"

# # Assign 120° spiral along a1 direction
# for i in mn_indices:
#     pos = supercell[i].scaled_position

#     # integer cell index along a1
#     n = int(np.floor(pos[0] * 3 + 1e-6))  # 0,1,2

#     phase = 2 * np.pi * n / 3

#     magmoms[i] = [
#         m * np.cos(phase),
#         m * np.sin(phase),
#         0.0
#     ]

# # Set moments
# supercell.set_initial_magnetic_moments(magmoms)

# # --- Debug print ---
# for i in mn_indices:
#     print(i, supercell[i].scaled_position, magmoms[i])


0 [0.  0.  0.5] [4.5 0.  0. ]
3 [0.33333333 0.         0.5       ] [-2.25        3.89711432  0.        ]
6 [0.66666667 0.         0.5       ] [-2.25       -3.89711432  0.        ]


In [9]:
# view(supercell)


## Second attempt at supercell

I think the above would not give the same bands.

Note that the supercell is different from the one in the paper.
In the paper they use $Q = (1/3,1/3)$

I think the above corresponds to $Q=(1/3,0)$

IT is the same local 120 degree spin structure. but it is not the same supercell, so the bands might differ.

Trying to create the one they use.

In [19]:
#Define transformation matrix from primitive to magnetic cell
P = np.array([
    [2, 1, 0],
    [-1, 1, 0],
    [0, 0, 1]
])

#defines supercell basis as linear combination of primitive basis
# So Supercell A1 = 2a_1+1a_2
#A2 = 1a_1+2a_2

# ERROR SHOULD BE A2=-a1+a2

Corrected to different supercell above now (initial one didn't converge anyway)

In [20]:
from ase.build import make_supercell

supercell = make_supercell(primitive, P)

In [21]:
view(supercell)

<Popen: returncode: None args: ['/home/runeekman/Desktop/DFT-project-2026/.v...>

In [22]:
magmoms = np.zeros((len(supercell), 3))
m = 4.5

# Get fractional coords in primitive basis
# Convert supercell scaled positions → primitive coordinates
# Using inverse transpose of P

P2D = P[:2, :2]
invP = np.linalg.inv(P2D)

for i, atom in enumerate(supercell):
    if atom.symbol != 'Mn':
        continue

    s = atom.scaled_position[:2]
    #This gives us r=s_1A_1 + s_2A_2 (i.e. in SUPERCELL lattice vectors)

    # map back to primitive coordinates
    r_prim = invP @ s

    phase = 2 * np.pi * (r_prim[0] / 3 + r_prim[1] / 3)
    #See the paper, the phase is given as Q dot r. So Q dot r = a_1*(1/3) + a_2*(1/3)

    #Now assigning the magnetic moments
    magmoms[i] = [
        m * np.cos(phase),
        m * np.sin(phase),
        0.0
    ]

supercell.set_initial_magnetic_moments(magmoms)

In [15]:
# view(supercell)

## GPAW calculator

In [26]:
# --- 4. Setup GPAW Calculator ---
# Article specifies: LDA functional, 600 eV cutoff.
# Symmetry must be off for non-collinear spirals.
# k-points: The magnetic BZ is smaller. If primitive was 12x12x1, supercell needs 4x12x1?
# Let's use a dense grid to resolve the bands smoothly for Fig 1.
calc = GPAW(
    mode={'name':'pw',
          'ecut':200},          # 600 eV cutoff in per paper. Rough first guess
    xc='LDA',              # Paper explicitly uses LDA (Ref 19)
    kpts={'size':(3,3,1), 'gamma':True},       # Adjusted for the rhombus supercell
    symmetry='off',        # Crucial for spiral
    magmoms=magmoms,       # Enforce non-collinear start
    spinpol=True,          # Needed for non-collinear
    occupations=FermiDirac(0.01),
    txt='MnI2_spiral_supercell.txt',
    parallel={'domain': 1, 'band': 1},
    maxiter=25,
)

calc.verbosity=1

supercell.calc = calc

### Run SCF calc

In [27]:
# --- 5. Run Calculation ---
print("Running SCF for magnetic supercell...")
energy = supercell.get_potential_energy()
calc.write('MnI2_spiral_gs.gpw')

Running SCF for magnetic supercell...


KeyboardInterrupt: 

In [ ]:
#Checking convergence and magnetic moments
energy = supercell.get_potential_energy()
print(energy) #check for divergence. If it diverges, something wrong in the setup

#Magmoms
print(supercell.get_magnetic_moments())
#We want three Mn moments, roughly 120 degrees apart, magnitude around 3-5 µB

#Total M
print(supercell.get_magnetic_moment())
#Should be 0. If they align ferromagnetically, phase may be incorrectly assigned

## Bandstructure

We are working in the magnetic supercell, so I guess(?) we will compute bands in the magnetic Brillouin Zone.

In [ ]:
from ase.dft.kpoints import bandpath

In [ ]:
M = [0.5, 0.0, 0.0]
minus_M = [-0.5, 0.0, 0.0]
Gamma = [0.0, 0.0, 0.0]

kpts = np.array([
    minus_M,
    Gamma,
    M
])

path = bandpath(kpts, supercell.cell, npoints=100)

In [ ]:
#Non SCF band calc
calc = GPAW('MnI2_gs.gpw').fixed_density(
    kpts=path,
    symmetry='off'
)

In [ ]:
bs = calc.band_structure()

In [ ]:
import matplotlib.pyplot as plt

bs.plot(emin=-3, emax=3)
plt.show()